In [ ]:
!nvidia-smi || true

!pip install torch torchvision lightning torchmetrics timm

Tue Feb 24 06:41:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   40C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split, Subset
from torchvision import datasets, transforms

In [ ]:
import pytorch_lightning as pl
import torch, random, os
import numpy as np

SEED = 42
pl.seed_everything(SEED, workers=True)

INFO:lightning_fabric.utilities.seed:Seed set to 42


42

In [ ]:
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    print("cuda:", torch.version.cuda)


torch: 2.10.0+cu128
cuda available: True
gpu: NVIDIA L4
cuda: 12.8


In [ ]:
from __future__ import annotations

import math
from dataclasses import dataclass
from typing import Optional, Sequence

import torch
from PIL import Image
from torch.utils.data import DataLoader, Subset
from torchvision import transforms
from torchvision.datasets import ImageFolder
import lightning.pytorch as pl


class RGBImageFolder(ImageFolder):
    def __getitem__(self, index):
        path, target = self.samples[index]
        sample = self.loader(path)

        # Force RGB if it's a PIL image and not already RGB (e.g., "L", "RGBA", etc.)
        if isinstance(sample, Image.Image) and sample.mode != "RGB":
            sample = sample.convert("RGB")

        if self.transform is not None:
            sample = self.transform(sample)
        if self.target_transform is not None:
            target = self.target_transform(target)

        return sample, target


def _resolve_count(n_total: int, split: float | int) -> int:
    if isinstance(split, float):
        return int(round(n_total * split))
    return int(split)


def _group_indices_by_class(targets: Sequence[int]) -> dict[int, list[int]]:
    class_to_idx: dict[int, list[int]] = {}
    for i, y in enumerate(targets):
        class_to_idx.setdefault(int(y), []).append(i)
    return class_to_idx


def _allocate_counts_proportional(
    caps: dict[int, int],
    n_total_needed: int,
    generator: torch.Generator,
) -> dict[int, int]:
    """
    Allocate integer counts per class summing to n_total_needed, approximately
    proportional to caps[c], and never exceeding caps[c].

    This uses floor + largest remainder, with deterministic tie-breaking plus
    a seeded shuffle to avoid systematic bias.
    """
    classes = list(caps.keys())
    cap_sum = sum(caps.values())
    if n_total_needed < 0 or n_total_needed > cap_sum:
        raise ValueError(f"Cannot allocate {n_total_needed} items from capacity {cap_sum}.")

    if n_total_needed == 0:
        return {c: 0 for c in classes}

    # Desired fractional counts
    desired = {c: (caps[c] * (n_total_needed / cap_sum)) for c in classes}

    # Base allocation = floor(desired), clipped to capacity
    counts = {c: min(int(math.floor(desired[c])), caps[c]) for c in classes}
    current = sum(counts.values())
    need = n_total_needed - current
    if need == 0:
        return counts

    # Remainders for distributing the remaining 'need'
    # To avoid always favoring lower class ids on ties, apply a seeded shuffle before sorting.
    perm = torch.randperm(len(classes), generator=generator).tolist()
    classes_shuffled = [classes[i] for i in perm]

    remainders = []
    for c in classes_shuffled:
        if counts[c] < caps[c]:
            remainders.append((desired[c] - counts[c], c))
    remainders.sort(reverse=True, key=lambda x: x[0])

    # First pass: give +1 by largest remainder
    i = 0
    while need > 0 and i < len(remainders):
        _, c = remainders[i]
        if counts[c] < caps[c]:
            counts[c] += 1
            need -= 1
        i += 1

    # If still need > 0, proportions are infeasible due to capacity clipping.
    # Fill remaining need deterministically from any class with capacity.
    if need > 0:
        fill_order = [c for c in classes_shuffled if counts[c] < caps[c]]
        j = 0
        while need > 0:
            c = fill_order[j % len(fill_order)]
            if counts[c] < caps[c]:
                counts[c] += 1
                need -= 1
            j += 1

    return counts


def _build_stratified_indices(
    targets: Sequence[int],
    n_val: int,
    n_test: int,
    seed: int,
) -> tuple[list[int], list[int], list[int]]:
    """
    Stratified split into train/val/test with exact sizes (when feasible),
    without falling back to non-stratified global random.
    """
    n_total = len(targets)
    if n_val + n_test >= n_total:
        raise ValueError("Need at least 1 sample left for training.")

    gen = torch.Generator().manual_seed(seed)
    class_to_indices = _group_indices_by_class(targets)

    # Shuffle each class's indices deterministically
    shuffled_per_class: dict[int, list[int]] = {}
    for c, idxs in class_to_indices.items():
        idxs_t = torch.tensor(idxs, dtype=torch.int64)
        perm = torch.randperm(len(idxs_t), generator=gen)
        shuffled_per_class[c] = idxs_t[perm].tolist()

    # 1) Allocate VAL counts proportional to original class sizes
    caps_val = {c: len(shuffled_per_class[c]) for c in shuffled_per_class}
    val_counts = _allocate_counts_proportional(caps_val, n_val, gen)

    # 2) Allocate TEST counts proportional to remaining capacity after val
    caps_after_val = {c: caps_val[c] - val_counts[c] for c in caps_val}
    test_counts = _allocate_counts_proportional(caps_after_val, n_test, gen)

    # 3) Slice indices
    train_idx, val_idx, test_idx = [], [], []
    for c, idxs in shuffled_per_class.items():
        v = val_counts[c]
        t = test_counts[c]
        val_part = idxs[:v]
        test_part = idxs[v : v + t]
        train_part = idxs[v + t :]

        val_idx.extend(val_part)
        test_idx.extend(test_part)
        train_idx.extend(train_part)

    # Sanity: exact sizes
    if not (len(val_idx) == n_val and len(test_idx) == n_test and len(train_idx) == n_total - n_val - n_test):
        raise RuntimeError(
            f"Split size mismatch: train={len(train_idx)}, val={len(val_idx)}, test={len(test_idx)} "
            f"(expected train={n_total - n_val - n_test}, val={n_val}, test={n_test})"
        )

    return train_idx, val_idx, test_idx


class FolderDataModule(pl.LightningDataModule):
    def __init__(
        self,
        train_dir: str,
        test_dir: str | None = None,
        train_transform: transforms.Compose | None = None,
        test_transform: transforms.Compose | None = None,
        batch_size: int = 256,
        num_workers: int = 4,
        val_split: float | int = 0.2,   # fraction or count (from train_dir)
        test_split: float | int = 0.1,  # fraction or count (ONLY used when test_dir is None)
        seed: int = 42,
        stratify: bool = True,
        pin_memory: bool = True,
        persistent_workers: bool = True,
        dataset_cls=None,
    ):
        super().__init__()
        if dataset_cls is None:
            dataset_cls = RGBImageFolder

        self.save_hyperparameters(ignore=["train_transform", "test_transform", "dataset_cls"])

        self.train_dir = train_dir
        self.test_dir = test_dir
        self.train_transform = train_transform
        self.test_transform = test_transform
        self.dataset_cls = dataset_cls

        self.train_ds = None
        self.val_ds = None
        self.test_ds = None

        self._indices_built = False
        self.train_idx: Optional[list[int]] = None
        self.val_idx: Optional[list[int]] = None
        self.test_idx: Optional[list[int]] = None

    def _ensure_indices(self):
        if self._indices_built:
            return

        base = self.dataset_cls(root=self.train_dir, transform=None)
        targets = base.targets
        n_total = len(targets)

        has_val = self.hparams.val_split_name is not None
        if has_val:
            n_val = 0
        else:
            n_val = _resolve_count(n_total, self.hparams.val_split)
            n_val = max(0, min(n_val, n_total - 1))

        if self.hparams.test_split_name is None:
            n_test = _resolve_count(n_total, self.hparams.test_split)
            n_test = max(0, min(n_test, n_total - n_val - 1))
        else:
            n_test = 0

        if self.test_dir is None:
            n_test = _resolve_count(n_total, self.hparams.test_split)
            n_test = max(0, min(n_test, n_total - n_val - 1))
        else:
            # external test set; only split train_dir into train/val
            n_test = 0

        if self.hparams.stratify:
            train_idx, val_idx, test_idx = _build_stratified_indices(
                targets=targets,
                n_val=n_val,
                n_test=n_test,
                seed=self.hparams.seed,
            )
        else:
            gen = torch.Generator().manual_seed(self.hparams.seed)
            perm = torch.randperm(n_total, generator=gen).tolist()
            val_idx = perm[:n_val]
            test_idx = perm[n_val : n_val + n_test]
            train_idx = perm[n_val + n_test :]

        self.train_idx, self.val_idx, self.test_idx = train_idx, val_idx, test_idx
        self._indices_built = True

    def setup(self, stage=None):
        self._ensure_indices()

        if stage in (None, "fit", "validate"):
            train_full = self.dataset_cls(root=self.train_dir, transform=self.train_transform)
            eval_full = self.dataset_cls(root=self.train_dir, transform=self.test_transform)

            self.train_ds = Subset(train_full, self.train_idx)
            self.val_ds = Subset(eval_full, self.val_idx)

        if stage in (None, "test"):
            if self.test_dir is not None:
                # NOTE: assumes class folders in test_dir match train_dir.
                self.test_ds = self.dataset_cls(root=self.test_dir, transform=self.test_transform)
            else:
                eval_full = self.dataset_cls(root=self.train_dir, transform=self.test_transform)
                self.test_ds = Subset(eval_full, self.test_idx)

    def train_dataloader(self):
        return DataLoader(
            self.train_ds,
            batch_size=self.hparams.batch_size,
            shuffle=True,
            num_workers=self.hparams.num_workers,
            pin_memory=self.hparams.pin_memory,
            persistent_workers=self.hparams.persistent_workers and self.hparams.num_workers > 0,
        )

    def val_dataloader(self):
        return DataLoader(
            self.val_ds,
            batch_size=self.hparams.batch_size,
            shuffle=False,
            num_workers=self.hparams.num_workers,
            pin_memory=self.hparams.pin_memory,
            persistent_workers=self.hparams.persistent_workers and self.hparams.num_workers > 0,
        )

    def test_dataloader(self):
        return DataLoader(
            self.test_ds,
            batch_size=self.hparams.batch_size,
            shuffle=False,
            num_workers=self.hparams.num_workers,
            pin_memory=self.hparams.pin_memory,
            persistent_workers=self.hparams.persistent_workers and self.hparams.num_workers > 0,
        )